# Xử lý dữ liệu CO2 từ dữ liệu NetCDF4 chuyển sang dữ liệu CSV tại Việt Nam
Đọc file NetCDF4 chứa dữ liệu CO2, lọc các điểm nằm trong lãnh thổ Việt Nam và chỉ giữ lại dữ liệu có chất lượng tốt chuyển sang file CSV


### Khai báo thư viện


In [ ]:
import netCDF4 # Thư viện để đọc file NetCDF4
from netCDF4 import num2date # Hàm chuyển đổi thời gian từ NetCDF
import pandas as pd # Thư viện để xử lý dữ liệu dạng bảng
import geopandas as gpd # Thư viện để xử lý dữ liệu địa lý
from datetime import datetime, timedelta # Thư viện để xử lý thời gian
import numpy as np # Thư viện tính toán số học
import os # Thư viện làm việc với hệ thống file
import glob # Thư viện để tìm kiếm file theo mẫu

In [ ]:
# Ranh giới shapefile Việt Nam dùng để lọc các CO2 trong lãnh thổ Việt Nam
shapefile_path = "E:\\RanhGioi\\VNM_adm\\gadm41_VNM_0.shp" 

### Hàm xử dữ liệu CO2 từ file NetCDF4 cho Việt Nam và cờ chất lượng tốt


In [ ]:
def process_single_nc_file(file_path, output_folder, shapefile_path):
    """
    Xử lý một file NetCDF4, lọc dữ liệu CO2 cho Việt Nam với chất lượng tốt
    
    Tham số:
        file_path: Đường dẫn đến file .nc4 cần xử lý
        output_folder: Thư mục lưu file CSV kết quả
        shapefile_path: Đường dẫn đến shapefile ranh giới Việt Nam
    
    Returns:
        True nếu xử lý thành công, False nếu có lỗi
    """
    try:
        # Bước 1: Mở file NetCDF4
        with netCDF4.Dataset(file_path, 'r') as nc_file:

            # Bước 2: Đọc các biến cần thiết từ file NetCDF4
            required_variables = [
                'latitude', 'longitude', 'time', 'date',
                'xco2_quality_flag', 'xco2'
            ]
            
            # Kiểm tra các biến có tồn tại không
            available_variables = nc_file.variables.keys()
            missing_vars = [var for var in required_variables if var not in available_variables]
            
            # Nếu thiếu biến, in cảnh báo và trả về False
            if missing_vars:
                print(f"⚠️ File thiếu biến: {missing_vars} - {os.path.basename(file_path)}")
                return False
            
            # Đọc dữ liệu từ các biến
            latitude = nc_file.variables['latitude'][:]
            longitude = nc_file.variables['longitude'][:]
            time_var = nc_file.variables['time']
            time_data = time_var[:]
            date_data = nc_file.variables['date'][:]
            xco2_quality_flag_data = nc_file.variables['xco2_quality_flag'][:]
            xco2_data = nc_file.variables['xco2'][:]
            
            # Bước 3: Tạo tên file CSV đầu ra
            output_csv_name = None # tạo một biến rỗng ban đầu để lưu tên file CSV đầu ra
            # Kiểm tra dữ liệu ngày tháng để tạo tên file 
            ## date_data.shape đảm bảo có ít nhất 1 dòng dữ liệu 
            ## date_data.ndim >= 2: xác nhận rằng date_data là mảng 2 chiều
            ## date_data.shape[1] >= 3: đảm bảo có ít nhất 3 cột (thường là năm, tháng, ngày)
            if date_data.shape[0] > 0 and date_data.ndim >= 2 and date_data.shape[1] >= 3:
                year_fn = int(date_data[0, 0])
                month_fn = int(date_data[0, 1])
                day_fn = int(date_data[0, 2])
                date_for_filename = f"{year_fn}-{month_fn:02d}-{day_fn:02d}"
                output_csv_name = f"raw_data_{date_for_filename}.csv"
            else:
                base_name = os.path.splitext(os.path.basename(file_path))[0]
                output_csv_name = f"{base_name}_processed_quality_filtered.csv"
            
            output_csv_path = os.path.join(output_folder, output_csv_name)

            # Bước 4: Chuyển đổi thời gian
            time_units = getattr(time_var, 'units', 'seconds since 1970-01-01 00:00:00')
            datetime_values_flat = []
            flat_time_data = time_data.flatten()

            if 'since' in time_units:
                try:
                    time_origin_str_full = time_units.split(' since ')[-1]
                    time_origin = datetime.strptime(time_origin_str_full, '%Y-%m-%d %H:%M:%S')
                    
                    if 'hours' in time_units.lower():
                        time_deltas_flat = [timedelta(hours=float(t)) for t in flat_time_data]
                    elif 'minutes' in time_units.lower():
                        time_deltas_flat = [timedelta(minutes=float(t)) for t in flat_time_data]
                    elif 'days' in time_units.lower():
                        time_deltas_flat = [timedelta(days=float(t)) for t in flat_time_data]
                    else:
                        time_deltas_flat = [timedelta(seconds=float(t)) for t in flat_time_data]
                    
                    datetime_values_flat = [time_origin + delta for delta in time_deltas_flat]
                except Exception as e_time:
                    print(f"⚠️ Lỗi chuyển đổi thời gian: {e_time}")
                    datetime_values_flat = flat_time_data.tolist()
            else:
                datetime_values_flat = flat_time_data.tolist()
            
            # Bước 5: Chuyển đổi dữ liệu ngày tháng
            dates_str_for_column = []
            if date_data.ndim == 2 and date_data.shape[0] == len(latitude.flatten()) and date_data.shape[1] >= 3:
                for i in range(date_data.shape[0]):
                    year = int(date_data[i, 0])
                    month = int(date_data[i, 1])
                    day = int(date_data[i, 2])
                    dates_str_for_column.append(f"{year}-{month:02d}-{day:02d}")
            else:
                dates_str_for_column = [None] * len(latitude.flatten())

            # Bước 6: Làm phẳng các mảng dữ liệu
            lat_flat = latitude.flatten()
            lon_flat = longitude.flatten()
            xco2_flat = xco2_data.flatten()
            xco2_quality_flag_flat = xco2_quality_flag_data.flatten()

            # Bước 7: Tạo GeoDataFrame
            points_gdf = gpd.GeoDataFrame(
                geometry=gpd.points_from_xy(lon_flat, lat_flat),
                crs="EPSG:4326"
            )
            points_gdf['original_index'] = np.arange(len(lat_flat))
            points_gdf['xco2_quality_flag'] = xco2_quality_flag_flat

            # Bước 8: Đọc shapefile Việt Nam
            vietnam_gdf = gpd.read_file(shapefile_path)
            if vietnam_gdf.crs != points_gdf.crs:
                vietnam_gdf = vietnam_gdf.to_crs(points_gdf.crs)

            # Bước 9: Lọc các điểm trong Việt Nam
            points_in_vietnam_gdf = gpd.sjoin(points_gdf, vietnam_gdf, how="inner", predicate="within")
            if points_in_vietnam_gdf.empty:
                print(f"⚠️ Không có điểm nào trong Việt Nam - {os.path.basename(file_path)}")
                return False
            
            # Bước 10: Lọc chất lượng tốt
            good_quality_points_df = points_in_vietnam_gdf[points_in_vietnam_gdf['xco2_quality_flag'] == 0].copy()
            if good_quality_points_df.empty:
                print(f"⚠️ Không có điểm chất lượng tốt - {os.path.basename(file_path)}")
                return False
            
            filtered_indices = good_quality_points_df['original_index'].values

            # Bước 11: Tạo DataFrame và xuất CSV
            df_data = {
                'latitude': lat_flat[filtered_indices],
                'longitude': lon_flat[filtered_indices],
                'time': [datetime_values_flat[i] for i in filtered_indices],
                'date': [dates_str_for_column[i] for i in filtered_indices],
                'xco2': xco2_flat[filtered_indices],
                'xco2_quality_flag': xco2_quality_flag_flat[filtered_indices],
            }
            df = pd.DataFrame(df_data)
            df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
            print(f"✅ Đã xử lý: {os.path.basename(file_path)} -> {len(df)} điểm")
            return True
            
    except OSError as e:
        print(f"❌ Lỗi đọc file (có thể file bị hỏng): {os.path.basename(file_path)}")
        print(f"   Chi tiết: {str(e)}")
        return False
    except Exception as e:
        print(f"❌ Lỗi không xác định: {os.path.basename(file_path)}")
        print(f"   Chi tiết: {str(e)}")
        return False

### Hàm main: Tự động xử lý toàn bộ file NetCDF4 và ghi log kết quả

In [ ]:
def main():
    # Thư mục chứa file NetCDF4
    input_folder = "E:\\DownloadData\\co2_nasa\\data_raw\\nc4\\oco2_nc4_2022"
    output_folder = "E:\\DownloadData\\co2_nasa\\data_processed\\csv_daily\\2022\\oco2"
    os.makedirs(output_folder, exist_ok=True)

    # Tìm tất cả file .nc4
    nc_files_to_process = glob.glob(os.path.join(input_folder, "*.nc4"))
    total_files = len(nc_files_to_process)
    
    print(f"🔍 Tìm thấy {total_files} file .nc4")
    print("=" * 60)
    
    success_count = 0
    error_count = 0
    error_files = []  # Danh sách file lỗi
    success_files = []  # Danh sách file thành công
    
    # Xử lý từng file
    for idx, nc_file_path in enumerate(nc_files_to_process, 1):
        filename = os.path.basename(nc_file_path)
        print(f"\n[{idx}/{total_files}] Đang xử lý: {filename}")
        
        if process_single_nc_file(nc_file_path, output_folder, shapefile_path):
            success_count += 1
            success_files.append(filename)
        else:
            error_count += 1
            error_files.append(filename)
    
    # Lưu danh sách file lỗi
    if error_files:
        error_log_path = os.path.join(output_folder, "error_files.txt")
        with open(error_log_path, 'w', encoding='utf-8') as f:
            f.write("DANH SÁCH FILE LỖI\n")
            f.write("=" * 60 + "\n\n")
            for i, filename in enumerate(error_files, 1):
                f.write(f"{i}. {filename}\n")
        print(f"\n📝 Đã lưu danh sách file lỗi vào: {error_log_path}")
    
    # Lưu danh sách file thành công
    if success_files:
        success_log_path = os.path.join(output_folder, "success_files.txt")
        with open(success_log_path, 'w', encoding='utf-8') as f:
            f.write("DANH SÁCH FILE THÀNH CÔNG\n")
            f.write("=" * 60 + "\n\n")
            for i, filename in enumerate(success_files, 1):
                f.write(f"{i}. {filename}\n")
        print(f"📝 Đã lưu danh sách file thành công vào: {success_log_path}")
    
    print("\n" + "=" * 60)
    print(f"✅ Hoàn tất xử lý!")
    print(f"   - Thành công: {success_count}/{total_files}")
    print(f"   - Lỗi: {error_count}/{total_files}")
    
    # Hiển thị một số file lỗi nếu có
    if error_files and len(error_files) <= 10:
        print(f"\n❌ Các file lỗi:")
        for filename in error_files:
            print(f"   - {filename}")
    elif error_files:
        print(f"\n❌ {len(error_files)} file lỗi (xem chi tiết trong error_files.txt)")

In [ ]:
if __name__ == "__main__":
    main()

### Gộp dữ liệu raw_data hàng ngày thành một file CSV năm

In [ ]:

# Thư mục chứa các file raw_data
input_folder = "E:\\DownloadData\\co2_nasa\\data_processed\\csv_daily\\2022\\oco2"
output_file = "E:\\DownloadData\\co2_nasa\\data_processed\\csv_yearly\\2022\\oco2\\oco2_vn_qual0_2022.csv"

# Tìm tất cả các file có dạng raw_data_YYYY_MM_DD.csv
all_files = glob.glob(os.path.join(input_folder, "raw_data_*.csv"))

# Danh sách chứa các DataFrame
df_list = []

for file in all_files:
    df = pd.read_csv(file)
    df_list.append(df)

# Gộp tất cả các DataFrame thành một
merged_df = pd.concat(df_list, ignore_index=True)

# Xuất ra file CSV tổng
merged_df.to_csv(output_file, index=False)

print(f"✅ Đã gộp {len(all_files)} file thành công -> {output_file}")